# San Francisco Dataset Visualization

This notebook generates a GeoPandas-based overview map for the Cabspotting / San Francisco benchmark used in the manuscript.

It loads `../sf_dataset.csv`, fetches the official San Francisco boundary from an online ArcGIS GeoJSON service, overlays all GPS start/end points, and exports `../sf_results_hpc/sf_dataset_overview_map.png` for use in the paper.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("../sf_dataset.csv")
OUTPUT_PATH = Path("../sf_results_hpc_2/sf_dataset_overview_map.png")
SF_BOUNDARY_GEOJSON_URL = (
    "https://services1-nocdn.arcgis.com/0MSEUqKaxRlEPj5g/ArcGIS/rest/services/"
    "San_Francisco_WFL1/FeatureServer/6/query?where=1%3D1&outFields=*&returnGeometry=true&f=geojson"
)


def extract_xy(series: pd.Series) -> pd.DataFrame:
    return series.str.extract(r"POINT\(([-0-9.]+) ([-0-9.]+)\)").astype(float)


df = pd.read_csv(DATA_PATH, usecols=["trajectory", "timestamp", "start_point", "end_point"])
start_xy = extract_xy(df["start_point"])
end_xy = extract_xy(df["end_point"])
ids = df["trajectory"].astype(str).str.strip().str.rsplit("_", n=1).str[0]

stats = {
    "records": len(df),
    "taxis": ids.nunique(),
    "time_start": df["timestamp"].min(),
    "time_end": df["timestamp"].max(),
    "lon_min": float(start_xy[0].min()),
    "lon_max": float(start_xy[0].max()),
    "lat_min": float(start_xy[1].min()),
    "lat_max": float(start_xy[1].max()),
}

stats

In [ ]:
sf_boundary = gpd.read_file(SF_BOUNDARY_GEOJSON_URL).to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(8.5, 8))
sf_boundary.boundary.plot(ax=ax, color="#202020", linewidth=1.6, zorder=3)
ax.scatter(
    start_xy[0].to_numpy(),
    start_xy[1].to_numpy(),
    s=0.12,
    alpha=0.05,
    color="#1f77b4",
    linewidths=0,
    rasterized=True,
    zorder=1,
    label="Trip start GPS points",
)
ax.scatter(
    end_xy[0].to_numpy(),
    end_xy[1].to_numpy(),
    s=0.12,
    alpha=0.04,
    color="#d62728",
    linewidths=0,
    rasterized=True,
    zorder=2,
    label="Trip end GPS points",
)

bounds = sf_boundary.total_bounds
ax.set_xlim(bounds[0] - 0.01, bounds[2] + 0.01)
ax.set_ylim(bounds[1] - 0.01, bounds[3] + 0.01)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Cabspotting / San Francisco boundary and GPS points")
ax.grid(alpha=0.2)
ax.legend(loc="lower left")
ax.text(
    0.02,
    0.98,
    "\n".join([
        f"Records: {stats['records']:,}",
        f"Taxis: {stats['taxis']:,}",
        f"Time span: {stats['time_start']} to {stats['time_end']}",
        "Boundary: official San Francisco ArcGIS layer",
        "Original format: start/end WKT points per record",
    ]),
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9,
    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "#b0b0b0"},
)
fig.tight_layout()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=220, bbox_inches="tight")
plt.close(fig)

OUTPUT_PATH